In [2]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
github_token = secrets.get_secret("github_secret")

github_username = "shaambhavi-dubey"
repo_name = "gnn-upi"
repo_url = f"https://{github_token}@github.com/{github_username}/{repo_name}.git"

!git clone {repo_url}
!cd gnn-upi && git config user.email "25bit087@sot.pdpu.ac.in"
!cd gnn-upi && git config user.name "shaambhavi-dubey"

Cloning into 'gnn-upi'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 45 (delta 17), reused 31 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 579.38 KiB | 3.24 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [3]:
# saved graf from nb1 again
import pickle
import pandas as pd
import networkx as nx

base_path = "/kaggle/input/datasets/shaambhavidubey/gnn-synthetic-data/gnn-upi/data"

with open(f"{base_path}/synthetic_graph.pkl", "rb") as f:
    DirGr = pickle.load(f)

node_df = pd.read_csv(f"{base_path}/node_features.csv")
print(node_df.shape)

(10000, 5)


In [4]:
# in-degree and out-degree are already in node_df from notebook 01,
# add pagerank from graph feature

in_deg = dict(DirGr.in_degree())
out_deg = dict(DirGr.out_degree())
pagerank = nx.pagerank(DirGr, weight='amount')  # weighted by transaction amount

graph_stats = []
for n in DirGr.nodes():
    graph_stats.append({
        "node_id": n,
        "in_degree": in_deg[n],
        "out_degree": out_deg[n],
        "pagerank": pagerank[n]
    })

graph_stats_df = pd.DataFrame(graph_stats)
print(graph_stats_df.head())

   node_id  in_degree  out_degree  pagerank
0        0         79          76  0.001917
1        1         53          56  0.001760
2        2          7           8  0.000196
3        3         76          79  0.001547
4        4        181         194  0.004512


In [5]:
# add with the tabular from nb2 as this is also xgb onlu
amt_stat = []
for n in DirGr.nodes():
    incoming_m = [DirGr[u][n]['amount'] for u in DirGr.predecessors(n)]
    outgoing_m = [DirGr[n][t]['amount'] for t in DirGr.successors(n)]
    total = incoming_m + outgoing_m
    amt_stat.append({
        "node_id": n,
        "avg_amt": sum(total)/len(total) if total else 0,
        "max_amount": max(total) if total else 0,
    })

amount_df = pd.DataFrame(amt_stat)

tabular_plus_graph_df = (
    node_df[["node_id", "account_age_days", "label"]]
    .merge(amount_df, on="node_id")
    .merge(graph_stats_df, on="node_id")
)
print(tabular_plus_graph_df.shape)
print(tabular_plus_graph_df.head())

(10000, 8)
   node_id  account_age_days  label     avg_amt  max_amount  in_degree  \
0        0               115      0   84.971419      527.30         79   
1        1               490      0  107.677431     1049.31         53   
2        2              1626      0   88.812000      350.70          7   
3        3               633      0  139.980129     5559.58         76   
4        4               720      0  112.883147     4161.46        181   

   out_degree  pagerank  
0          76  0.001917  
1          56  0.001760  
2           8  0.000196  
3          79  0.001547  
4         194  0.004512  


In [6]:
# again doing same straifty train test split
from sklearn.model_selection import train_test_split

X = tabular_plus_graph_df.drop(columns=["node_id", "label"])
y = tabular_plus_graph_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train fraud: {y_train.sum()}, Test fraud: {y_test.sum()}")

Train fraud: 200, Test fraud: 50


In [7]:
from xgboost import XGBClassifier

scale = (y_train == 0).sum() / (y_train == 1).sum()

model_graphstats = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    scale_pos_weight=scale,
    eval_metric="aucpr",
    random_state=42
)
model_graphstats.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [8]:
from sklearn.metrics import classification_report, average_precision_score, f1_score

y_pred = model_graphstats.predict(X_test)
y_proba = model_graphstats.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"PR-AUC: {average_precision_score(y_test, y_proba):.3f}")
print(f"F1: {f1_score(y_test, y_pred):.3f}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1950
           1       0.96      0.98      0.97        50

    accuracy                           1.00      2000
   macro avg       0.98      0.99      0.98      2000
weighted avg       1.00      1.00      1.00      2000

PR-AUC: 0.999
F1: 0.970


In [9]:
# see if the graph features acc hold importance or nah
import pandas as pd

importances = pd.Series(model_graphstats.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

in_degree           0.957873
avg_amt             0.020219
max_amount          0.014538
out_degree          0.004553
pagerank            0.002627
account_age_days    0.000191
dtype: float32


In [10]:
results_03 = {
    "notebook": "03_baseline_xgboost_graphstats",
    "features_used": ["account_age_days", "avg_amt", "max_amount", "in_degree", "out_degree", "pagerank"],
    "precision_fraud": 0.96,
    "recall_fraud": 0.98,
    "f1_fraud": 0.970,
    "pr_auc": 0.999
}

import json
with open("gnn-upi/data/results_03_graphstats.json", "w") as f:
    json.dump(results_03, f, indent=2)

print("saved")

saved


In [11]:
!cd gnn-upi && git add . && git commit -m "Notebook 03: XGBoost + graph stats baseline (F1=0.97, PR-AUC=0.999)"
!cd gnn-upi && git pull origin main --no-edit
!cd gnn-upi && git push

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
From https://github.com/shaambhavi-dubey/gnn-upi
 * branch            main       -> FETCH_HEAD
Already up to date.
Everything up-to-date
